# Analiza wyników eksperymentów

In [3]:
import pandas as pd
import os

RESULTS_DIR = "../results/"
REQ_DIR_200 = RESULTS_DIR + "200_req/"
REQ_DIR_3000 = RESULTS_DIR + "3000_req/"

files_200_req = os.listdir(REQ_DIR_200)
files_3000_req = os.listdir(REQ_DIR_3000)

files_200_req_csv = [f for f in files_200_req if f.endswith('.csv')]
files_200_req_log = [f for f in files_200_req if f.endswith('.log')]
files_3000_req_csv = [f for f in files_3000_req if f.endswith('.csv')]
files_3000_req_log = [f for f in files_3000_req if f.endswith('.log')]

print("== Detected files ==")
print(f'200 req: {len(files_200_req_csv)} CSV, {len(files_200_req_log)} LOG')
print(f'3000 req: {len(files_3000_req_csv)} CSV, {len(files_3000_req_log)} LOG')

== Loaded files ==
200 req: 8 CSV, 8 LOG
3000 req: 8 CSV, 8 LOG


## Wczytanie plików .csv i .log

In [17]:
import csv
from typing import List, Dict

def read_log_with_csv(path: str) -> pd.DataFrame:
    rows = []
    with open(path, newline='', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                continue
            rows.append(line)

    return pd.DataFrame(rows, columns=['data'])

In [18]:
df_200_req_csv = [(pd.read_csv(REQ_DIR_200 + f), f) for f in files_200_req_csv]
df_3000_req_csv = [(pd.read_csv(REQ_DIR_3000 + f), f) for f in files_3000_req_csv]

df_200_req_log = [ (read_log_with_csv(REQ_DIR_200 + f), f) for f in files_200_req_log]
df_3000_req_log = [ (read_log_with_csv(REQ_DIR_3000 + f), f) for f in files_3000_req_log]

In [19]:
print("== loaded csv dataframes ==")
for df, fname in df_200_req_csv + df_3000_req_csv:
    print(f"{fname}: \t{df.shape}")

== loaded csv dataframes ==
docker_run_NB_fastapi_test_results.csv: 	(200, 6)
docker_run_NB_ray_test_results.csv: 	(200, 6)
docker_run_SVC_fastapi_test_results.csv: 	(200, 6)
docker_run_SVC_ray_test_results.csv: 	(200, 6)
local_run_NB_fastapi_test_results.csv: 	(200, 6)
local_run_NB_ray_test_results.csv: 	(200, 6)
local_run_SVC_fastapi_test_results.csv: 	(200, 6)
local_run_SVC_ray_test_results.csv: 	(200, 6)
docker_run_NB_fastapi_test_results.csv: 	(3000, 10)
docker_run_NB_ray_test_results.csv: 	(3000, 10)
docker_run_SVC_fastapi_test_results.csv: 	(3000, 10)
docker_run_SVC_ray_test_results.csv: 	(3000, 10)
local_run_NB_fastapi_test_results.csv: 	(3000, 10)
local_run_NB_ray_test_results.csv: 	(3000, 10)
local_run_SVC_fastapi_test_results.csv: 	(3000, 10)
local_run_SVC_ray_test_results.csv: 	(3000, 10)


In [20]:
for df, fname in df_200_req_csv + df_3000_req_csv:
    splits = fname.split('_')
    ml_alg_type = splits[2]
    server_type = splits[3]
    env_type = splits[0]
    print(f"{fname}: \t{df.shape} \t{env_type} \t{ml_alg_type} \t{server_type}")

    df['env_type'] = env_type
    df['ml_alg_type'] = ml_alg_type
    df['server_type'] = server_type


docker_run_NB_fastapi_test_results.csv: 	(200, 6) 	docker 	NB 	fastapi
docker_run_NB_ray_test_results.csv: 	(200, 6) 	docker 	NB 	ray
docker_run_SVC_fastapi_test_results.csv: 	(200, 6) 	docker 	SVC 	fastapi
docker_run_SVC_ray_test_results.csv: 	(200, 6) 	docker 	SVC 	ray
local_run_NB_fastapi_test_results.csv: 	(200, 6) 	local 	NB 	fastapi
local_run_NB_ray_test_results.csv: 	(200, 6) 	local 	NB 	ray
local_run_SVC_fastapi_test_results.csv: 	(200, 6) 	local 	SVC 	fastapi
local_run_SVC_ray_test_results.csv: 	(200, 6) 	local 	SVC 	ray
docker_run_NB_fastapi_test_results.csv: 	(3000, 10) 	docker 	NB 	fastapi
docker_run_NB_ray_test_results.csv: 	(3000, 10) 	docker 	NB 	ray
docker_run_SVC_fastapi_test_results.csv: 	(3000, 10) 	docker 	SVC 	fastapi
docker_run_SVC_ray_test_results.csv: 	(3000, 10) 	docker 	SVC 	ray
local_run_NB_fastapi_test_results.csv: 	(3000, 10) 	local 	NB 	fastapi
local_run_NB_ray_test_results.csv: 	(3000, 10) 	local 	NB 	ray
local_run_SVC_fastapi_test_results.csv: 	(3000, 10

In [52]:
import re

for logs, fname in df_200_req_log + df_3000_req_log:
    list_with_czas = [l for l in logs['data'] if 'Czas' in l]
    czas = [(m.group(1) if (m := re.search(r'(\d+(?:\.\d+)?ms)', line)) else None) for line in list_with_czas]
    print(czas[:2])
    print(f'{fname} - {len(list_with_czas)}')

    splits = fname.split('_' or '.')
    splits[-1] = splits[-1].split('.')[0]
    str_types = [s for s in splits if s in ['d', 'local', 'docker', 'svc', 'NB', 'SVC', 'nb', 'ray', 'fastapi']]

    for s in str_types:
        if s == 'nb':
            str_types[str_types.index(s)] = 'NB'
        if s == 'svc':
            str_types[str_types.index(s)] = 'SVC'
        if s == 'd':
            str_types[str_types.index(s)] = 'docker'

    print(str_types)
    # print(splits)

    df = [(df,f) for df, f in df_200_req_csv + df_3000_req_csv if all(s in f for s in str_types)]
    df = [(d,f) for d,f in df if 'csv' in f]
    # TODO: do poprawy nazewnictwo plikow
    #  (np. dodanie 200req lub 3000req do kazdego z pliku dla lepsze identyfikacji)
    for d,f in df:
        print(f'{f}')

    if df:
        df = df[0]
        if 'error_details' in df.columns:
            success_mask = df['error_details'].isna() | (df['error_details'] == "")
            num_successes = success_mask.sum()

            if num_successes == len(czas):
                df.loc[success_mask, 'log_czas'] = czas

        else:
            if len(df) == len(czas):
                df['log_czas'] = czas

['48.11ms', '48.20ms']
docker_run_NB_fastapi_test_results.log - 200
['docker', 'NB', 'fastapi']
docker_run_NB_fastapi_test_results.csv
docker_run_NB_fastapi_test_results.csv


AttributeError: 'tuple' object has no attribute 'columns'

In [34]:
df_200_req_csv[0][0]

,input_text,duration_seconds,status,text,prediction,model_used,env_type,ml_alg_type,server_type
0,Just as the the title says. I feel like one is...,0.2778,success,Just as the the title says. I feel like one is...,Depression,MultinomialNB,docker,NB,fastapi
1,a blackened sky encroached tugging behind it m...,0.1631,success,a blackened sky encroached tugging behind it m...,Depression,MultinomialNB,docker,NB,fastapi
2,"It gives you insomnia, which in turn makes you...",0.2523,success,"It gives you insomnia, which in turn makes you...",Depression,MultinomialNB,docker,NB,fastapi
3,"Hello all, I'm a new submitter to this channel...",0.1634,success,"Hello all, I'm a new submitter to this channel...",Depression,MultinomialNB,docker,NB,fastapi
4,Thank God the CB is over for Eid,0.2523,success,Thank God the CB is over for Eid,Normal,MultinomialNB,docker,NB,fastapi
...,...,...,...,...,...,...,...,...,...
195,i can't afford it.,0.2987,success,i can't afford it.,Depression,MultinomialNB,docker,NB,fastapi
196,27 m I never had a lot of friends in highschoo...,0.2987,success,27 m I never had a lot of friends in highschoo...,Depression,MultinomialNB,docker,NB,fastapi
197,I stopped smoking a couple days ago cuz my lun...,0.2987,success,I stopped smoking a couple days ago cuz my lun...,Depression,MultinomialNB,docker,NB,fastapi
198,a team of doctors gave her a whole new face.,0.2988,success,a team of doctors gave her a whole new face.,Normal,MultinomialNB,docker,NB,fastapi
